In [1]:
!pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 31.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os
import numpy as np
import open_clip
import torch
from torch import nn
import timm

In [14]:
class OpenClipViT(nn.Module):
    def __init__(self, model_name, pretrained, with_proj_layer):
        super().__init__()
        model, train_transform, _ = open_clip.create_model_and_transforms(model_name, pretrained)
        self.vit = model.visual

        if not with_proj_layer:
            self.vit.proj = None
            self.out_dim = self.vit.transformer.width
        else:
            self.out_dim = self.vit.output_dim

        self.img_size = self.vit.image_size[-1]
        self.mean = train_transform.transforms[-1].mean
        self.std = train_transform.transforms[-1].std

    def forward(self, x):
        embedding = self.vit(x)

        return embedding


class EVA02ViT(nn.Module):
    def __init__(self, model_name, with_proj_layer):
        super().__init__()
        if with_proj_layer:
            self.vit = timm.create_model(model_name, pretrained=True)
            self.out_dim = self.vit.num_classes
        else:
            self.vit = timm.create_model(model_name, pretrained=True, num_classes=0)
            self.out_dim = self.vit.embed_dim

        self.img_size = self.vit.default_cfg["input_size"][-1]
        self.mean = self.vit.default_cfg["mean"]
        self.std = self.vit.default_cfg["std"]

    def forward(self, x):
        embedding = self.vit(x)

        return embedding


class DINOv2ViT(nn.Module):
    def __init__(self, model_name, img_size=None):
        super().__init__()
        if img_size is None:
            self.vit = timm.create_model(model_name, pretrained=True)
            self.img_size = self.vit.default_cfg["input_size"][-1]
        else:
            self.vit = timm.create_model(model_name, pretrained=True, img_size=img_size)
            self.img_size = img_size

        self.out_dim = self.vit.embed_dim
        self.mean = self.vit.default_cfg["mean"]
        self.std = self.vit.default_cfg["std"]

    def forward(self, x):
        embedding = self.vit(x)

        return embedding


class SAMViT(nn.Module):
    def __init__(self, model_name, image_size=224):
        super().__init__()
        self.vit = timm.create_model(model_name, pretrained=True)
        self.vit.neck = torch.nn.Identity()
        self.out_dim = self.vit.embed_dim
        self.img_size = image_size
        self.mean = self.vit.default_cfg["mean"]
        self.std = self.vit.default_cfg["std"]

    def forward(self, x):
        embeddings = self.vit(x)

        return embeddings


class OpenClipConvNext(nn.Module):
    def __init__(self, model_name, pretrained, with_proj_layer):
        super().__init__()
        model, train_transform, _ = open_clip.create_model_and_transforms(model_name, pretrained)

        if with_proj_layer:
            self.encoder = model.visual
            try:
                self.out_dim = self.encoder.head.proj.out_features
            except AttributeError:
                self.encoder.head.mlp.drop2 = nn.Dropout(p=0.0)
                self.out_dim = self.encoder.head.mlp.fc2.out_features
        else:
            self.encoder = model.visual.trunk
            self.out_dim = self.encoder.num_features

        self.img_size = model.visual.image_size[0]
        self.mean = model.visual.image_mean
        self.std = model.visual.image_std

    def forward(self, x):
        embedding = self.encoder(x)

        return embedding


class SigLIPViT(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.vit = timm.create_model(model_name, pretrained=True)
        self.img_size = self.vit.default_cfg["input_size"][-1]
        self.out_dim = self.vit.num_features
        self.mean = self.vit.default_cfg["mean"]
        self.std = self.vit.default_cfg["std"]

    def forward(self, x):
        embedding = self.vit(x)

        return embedding


def get_foundational_model(config):
    if config.MODEL.BACKBONE.type in ["clip", "clipav2", "meta-clip"]:
        model = OpenClipViT(config.MODEL.BACKBONE.model_name, config.MODEL.BACKBONE.weights,
                            config.MODEL.BACKBONE.proj_layer)

    elif config.MODEL.BACKBONE.type == "eva02":
        model = EVA02ViT(config.MODEL.BACKBONE.model_name, config.MODEL.BACKBONE.proj_layer)

    elif config.MODEL.BACKBONE.type == "dinov2":
        model = DINOv2ViT(config.MODEL.BACKBONE.model_name, config.TRANSFORM.size)

    elif config.MODEL.BACKBONE.type == "sam":
        model = SAMViT(config.MODEL.BACKBONE.model_name, config.TRANSFORM.size)

    elif config.MODEL.BACKBONE.type == "clip_convnext":
        model = OpenClipConvNext(config.MODEL.BACKBONE.model_name, config.MODEL.BACKBONE.weights,
                                 config.MODEL.BACKBONE.proj_layer)

    elif config.MODEL.BACKBONE.type == "siglip":
        model = SigLIPViT(config.MODEL.BACKBONE.model_name)

    else:
        raise NotImplementedError(f"Unsupported foundational model: {config.MODEL.BACKBONE.type}")

    return model


class Neck(nn.Module):
    def __init__(self, config, in_features=None):
        super().__init__()
        if not None:
            self.in_features = in_features
        else:
            self.in_features = config.MODEL.BACKBONE.output_dim
        self.out_features = config.MODEL.embedding_dim
        self.neck_type = config.MODEL.NECK.neck_type
        self.dropout = config.MODEL.NECK.dropout

        if self.neck_type == "proj_layer":
            self.neck = nn.Sequential(
                nn.Dropout(self.dropout),
                nn.Linear(self.in_features, self.out_features)
            )
        elif self.neck_type == "pooling":
            self.neck = nn.AdaptiveAvgPool1d(self.out_features)
        else:
            raise NotImplementedError(f"Unsupported neck type: {self.neck_type}")

    def forward(self, x):
        return self.neck(x)


class Config:
      # Model configurations
      class MODEL:
          class BACKBONE:
              # Backbone model details
              type = "eva02"  # backbone type (could be "eva02", "dinov2", etc.)
              model_name = "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k"  # specify the model name
              proj_layer = True  # set to True if you want the projection layer

          class NECK:
              # Neck layer configurations
              neck_type = "proj_layer"  # "proj_layer" or "pooling"
              dropout = 0.1  # dropout rate

          embedding_dim = 256  # final embedding dimension (size of the output feature vector)

      # Image transformation configurations
      class TRANSFORM:
          size = 448

if __name__ == "__main__":
    main()

Processed 10 images
Processed 20 images
Processed 30 images
Processed 40 images
Processed 50 images
Processed 60 images
Processed 70 images
Processed 80 images
Processed 90 images
Processed 100 images
Processed 110 images
Processed 120 images
Processed 130 images
Processed 140 images
Processed 150 images
Processed 160 images
Processed 170 images
Processed 180 images
Processed 190 images
Processed 200 images
Processed 210 images
Processed 220 images
Processed 230 images
Processed 240 images
Processed 250 images
Processed 260 images
Processed 270 images
Processed 280 images
Processed 290 images
Processed 300 images
Processed 310 images
Processed 320 images
Processed 330 images
Processed 340 images
Processed 350 images
Processed 360 images
Processed 370 images
Processed 380 images
Processed 390 images
Processed 400 images
Processed 410 images
Processed 420 images
Processed 430 images
Processed 440 images
Processed 450 images
Processed 460 images
Processed 470 images
Processed 480 images
P

In [13]:
import timm
print(timm.list_models("*eva*"))

['eva02_base_patch14_224', 'eva02_base_patch14_448', 'eva02_base_patch16_clip_224', 'eva02_enormous_patch14_clip_224', 'eva02_large_patch14_224', 'eva02_large_patch14_448', 'eva02_large_patch14_clip_224', 'eva02_large_patch14_clip_336', 'eva02_small_patch14_224', 'eva02_small_patch14_336', 'eva02_tiny_patch14_224', 'eva02_tiny_patch14_336', 'eva_giant_patch14_224', 'eva_giant_patch14_336', 'eva_giant_patch14_560', 'eva_giant_patch14_clip_224', 'eva_large_patch14_196', 'eva_large_patch14_336']


In [12]:
def extract_features(image_path, model, transform):
    # Load and preprocess the image
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0)  # Add batch dimension

    # Pass the image through the model to extract features
    with torch.no_grad():  # Disable gradient tracking
        feature = model(image)  # Model inference

    return feature.cpu().numpy()

In [11]:
# Function to extract and save features for the entire dataset
def save_features(data_dir, feature_save_path, label_save_path, model, transform):
    all_features = []
    labels = []
    dem = 0

    # Iterate through subfolders (labels)
    for subfolder in os.listdir(data_dir):
        subfolder_path = os.path.join(data_dir, subfolder)
        if os.path.isdir(subfolder_path):
            image_files = [f for f in os.listdir(subfolder_path) if f.endswith(('jpg', 'jpeg', 'png'))]

            # Iterate through image files in the subfolder
            for image_file in image_files:
                image_path = os.path.join(subfolder_path, image_file)
                
                # Extract features for each image
                features = extract_features(image_path, model, transform)
                all_features.append(features)
                labels.append(subfolder)  # Save the subfolder name as the label
                dem += 1

                # Print progress every 10 images
                if dem % 10 == 0:
                    print(f"Processed {dem} images")

    # Convert the list of features to a numpy array
    all_features = np.vstack(all_features)  # Combine all features into one array

    # Save features to one file
    np.save(feature_save_path, all_features)
    print(f"Features saved to {feature_save_path}")

    # Save labels to a separate file
    np.save(label_save_path, np.array(labels))
    print(f"Labels saved to {label_save_path}")


In [10]:
def main():
    # Define data directory and save path for extracted features
    data_dir = "/kaggle/input/images/ImagesData"
    feature_save_path = "saved_features_OpenClipViT.npy"
    label_save_path = "labels_OpenClipViT.npy"

    transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

    # Set up configuration
    config = Config()
    model = get_foundational_model(config)

    # Initialize the model based on the config
    model = get_foundational_model(config)
    model.eval()  # Set the model to evaluation mode

    # Extract and save features for the entire dataset
    # Corrected call to save_features with all required arguments
    save_features(data_dir, feature_save_path, label_save_path, model, transform)


    print("Feature extraction completed and saved.")  # Notify completion